In [1]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import os, pickle

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker
import hist
import vector
import pandas as pd

from physics.simulation import msq, mcfm
# from physics.hstar import eft
from physics.analysis import zz4l
from datasets import coefficient
from models import taylr

In [2]:
import numpy as np
from numpy.polynomial import polynomial as P
from itertools import product

from physics.simulation import mcfm

class Modifier():

  def __init__(self, baseline, events, c6_values = [-20, -10, 0, 10, 20], ct_values = [-1,0,1], cg_values = [-0.01, 0.0, 0.01]):
    self.baseline = baseline
    self.events = events

    self.c6_values = np.array(c6_values)
    self.ct_values = np.array(ct_values)
    self.cg_values = np.array(cg_values)
    self.c6_degree = len(c6_values) - 1
    self.ct_degree = len(ct_values) - 1
    self.cg_degree = len(cg_values) - 1

    X, Y, Z = np.meshgrid(self.c6_values, self.ct_values, self.cg_values, indexing='ij')  # Shape: (5, 3, 3) 
    self.V = P.polyvander3d(X, Y, Z, [self.c6_degree, self.ct_degree, self.cg_degree])

    msq_sm  = self.events.components[mcfm.mcfm_component_sm[self.baseline]].to_numpy()
    xyz_bsm = []
    for i, c6_val in enumerate(self.c6_values):
      for j, ct_val in enumerate(self.ct_values):
        for k, cg_val in enumerate(self.cg_values):
          xyz_bsm.append((c6_val, ct_val, cg_val))
    xyz_npts = len(xyz_bsm)
    msq_bsm = self.events.components[[mcfm.mcfm_component_bsm[self.baseline][xyz] for xyz in xyz_bsm]].to_numpy()

    self.bsm_values = msq_bsm / msq_sm[:, np.newaxis]
    # self.bsm_values = self.bsm_values.reshape(self.bsm_values.shape[0], self.V.shape[0], self.V.shape[1], self.V.shape[2])

    V_matrix = self.V.reshape(xyz_npts, xyz_npts)
    b = self.bsm_values.reshape(-1, xyz_npts).T
    coefficients = np.linalg.solve(V_matrix, b)
    self.coefficients = coefficients.T.reshape(-1, self.c6_degree+1, self.ct_degree+1, self.cg_degree+1)

    # filter out non-physical coefficients
    # self.coefficients[:, 3, 2:, :] = 0
    # self.coefficients[:, 3, :, 2:] = 0
    # self.coefficients[:, 4, 1:, :] = 0
    # self.coefficients[:, 4, :, 1:] = 0

  def modify(self, c6 = None, ct = None, cg = None):
    c6_powers = np.stack([np.power(c6,i) for i in range(self.c6_degree+1)], axis=0)   # (5, Nx)
    ct_powers = np.stack([np.power(ct,j) for j in range(self.ct_degree+1)], axis=0)   # (3, Ny)
    cg_powers = np.stack([np.power(cg,k) for k in range(self.cg_degree+1)], axis=0)   # (3, Nz)
    msq_bsm_over_sm = np.einsum('nijk,ix,jy,kz->nxyz', self.coefficients, c6_powers, ct_powers, cg_powers)

    w_bsm = msq_bsm_over_sm * self.events.weights.to_numpy()[:, np.newaxis, np.newaxis, np.newaxis]
    p_bsm = w_bsm / np.sum(w_bsm, axis=0, keepdims=True)

    return w_bsm, p_bsm

In [19]:
csv_filepath = '/ptmp/mpp/taepa/higgs-offshell-interpretation/data/eft/zz4l/ggZZ_sig/events.csv'
events = mcfm.from_csv(cross_section = 1.0, file_path=csv_filepath, n_rows = 200_000)
events = zz4l.analyze(events)

In [20]:
l1 = vector.array({"pt": events.kinematics['l1_pt'], "eta": events.kinematics['l1_eta'], "phi": events.kinematics['l1_phi'], "energy": events.kinematics['l1_energy']})
l2 = vector.array({"pt": events.kinematics['l2_pt'], "eta": events.kinematics['l2_eta'], "phi": events.kinematics['l2_phi'], "energy": events.kinematics['l2_energy']})
l3 = vector.array({"pt": events.kinematics['l3_pt'], "eta": events.kinematics['l3_eta'], "phi": events.kinematics['l3_phi'], "energy": events.kinematics['l3_energy']})
l4 = vector.array({"pt": events.kinematics['l4_pt'], "eta": events.kinematics['l4_eta'], "phi": events.kinematics['l4_phi'], "energy": events.kinematics['l4_energy']})

m4l = (l1 + l2 + l3 + l4).mass

In [21]:
# eft_mod = Modifier(baseline = msq.Component.SBI, events=events, c6_values = [-20,-10, 0, 10, 20], ct_values = [-1,0,1], cg_values = [-0.01, 0,0.01])
eft_mod = Modifier(baseline = msq.Component.SIG, events=events, c6_values = [-20,-10, 0, 10, 20], ct_values = [-1,0,1], cg_values = [-0.01, 0,0.01])
# eft_mod = Modifier(baseline = msq.Component.INT, events=events, c6_values = [-20, 0, 20], ct_values = [-1,1], cg_values = [-0.01, 0.01])

In [ ]:
lw = 1
color_sm = 'black'
line_sm = '--'

# bsm_wc = 'c6'
# bsm_vals = [-18.7574, 18.7574]

bsm_wc = 'ct'
bsm_vals = [-1.27167, 1.27617]

# bsm_wc = 'cg' 
# bsm_vals = [-1.35156, 1.35156]

bsm_colors = ['red', 'blue']
bsm_lines = ['-', '-']

w_bsm, _ = eft_mod.modify(c6=bsm_vals if bsm_wc == 'c6' else [0], ct=bsm_vals if bsm_wc == 'ct' else [0], cg=bsm_vals if bsm_wc == 'cg' else [0])
w_bsm = w_bsm.reshape(-1, len(bsm_vals))

In [39]:
xobs = events.kinematics['l1_pt'].to_numpy()
nxbins = 38
xmin, xmax = 20.0, 400.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [40]:
h_sbi_sm = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sbi_sm.fill(xobs, weight=events.weights)

h_sbi_bsm = []
for i_bsm, bsm_val in enumerate(bsm_vals):
    h = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
    h.fill(xobs, weight=w_bsm[:,i_bsm])
    h_sbi_bsm.append(h)


In [41]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)

l_bsm = []
for i_bsm, bsm_val in enumerate(bsm_vals):
    l_bsm.append(ax1.stairs(h_sbi_bsm[i_bsm].values(), xbins, color=bsm_colors[i_bsm], linestyle=bsm_lines[i_bsm], linewidth=lw))
l_sm = ax1.stairs(h_sbi_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

l_sm.set_label('$\mathrm{SM}$')
for i_bsm, bsm_val in enumerate(bsm_vals):
    sgn = '-' if bsm_val < 0 else '+'
    l_bsm[i_bsm].set_label(f'$c = {sgn}{abs(bsm_val)}$')
ax1.legend(frameon=False, fontsize=12)

for i_bsm, bsm_val in enumerate(bsm_vals):
    ax2.stairs(h_sbi_bsm[i_bsm].values() / h_sbi_sm.values(), xbins, color=bsm_colors[i_bsm], linestyle=bsm_lines[i_bsm], linewidth=lw)
ax2.stairs(h_sbi_sm.values() / h_sbi_sm.values(), xbins, color=color_sm, linestyle=line_sm, linewidth=lw)

# ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} m_{ZZ}\\ \\mathrm{[fb / GeV]}$', loc='top', fontsize=15)
# ax1.set_ylim(0.0, 1.0)

# ax1.text(0.04 ,0.12, '$gg (\\rightarrow h^{\\ast}) \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)
# ax1.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)

ax2.set_ylabel('$\\mathrm{BSM} / \\mathrm{SM}$', fontsize=15)
# ax2.set_ylim(0.0,5.0)

ax2.set_xscale('linear')
ax2.set_xlim(xmin, xmax)
# ax2.set_xlabel('mZZ', loc='right', fontsize=15)
ax2.set_xlabel('pTl1', loc='right', fontsize=15)
ax2.tick_params(top=True, labeltop=False)

fig.tight_layout()
fig.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

# plt.savefig('sbi_4l_mzz.pdf')
plt.show()
plt.close()